<a href="https://colab.research.google.com/github/r1shi-j/coa122-coursework/blob/main/122_coursework.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

### System Skeleton
Show main menu
- check availability of tools
  - search for tools
- rent tools to customers
  - must have valid subscription
  - must have not reached their renting limit
  - tool must be available (and a valid tool)
  - return tools
  - search for tools
- maintain a database of all tools
  - search for tools
- pruning
  - reads Tool_Feedback to understand which tools aren't popular
-----------------------------------------------------
_Files to use_

`Tool_Info.txt`
- stores ToolID,Name,Size,PurchasePrice,PurchaseDate
- all tools
- read-only
  - list all tools (filter by available and search term)
  - check if tool exists (helper)

`Tool_Feedback.txt`
- stores ToolID,Rating,Comments
- logs customer feedback about the tool
- read-append
  - log tool feedback
  - find the least popular tools

`Subscription_Info.txt`
- stores CustomerID,SubscriptionType,StartDate,EndDate
- all customers with a subscription
- read-only
  - check if customer has a subscription (helper)
  - check if customer has reached their renting limit (helper)

`Rental_History.txt`
- stores ToolID,CustomerID,RentalDate,ReturnDate
- logs all rentals
- read-append
  - check if tool is available (helper)
  - log a tool rental
  - log a tool return

`Booking.txt`
- stores DIYClass,BookingDate,Time
- all DIY class bookings?
- read-only?

-----------------------------------------------------

Make dummy data
- generate_id function for 4-char UUID

```each tool has a toolID
but each tool has multiple copies, copyID
so when renting tools we find the toolID, and then check each copyID if it is available, store a count of x/y available in the tool row
When returning tools we have to match the toolID and customerID... to find the exact tool and copy they rented, or maybe create a rentID "ticket" given to a customer.

each physically unique tool has a unique toolID, so to collect a list of all copies of a tool, we create loadGroupedTools which aggregates the toolIDs of tools with the same name, in addition it creates a list of availableIDs ready to show a count of how many instances of that tool are available.
```

In [36]:
import subscriptionManager as sm
import feedbackManager as fm
from datetime import datetime

In [37]:
### TEMP

# Check Feedback Manager Works.
feedback_list = fm.load_feedback(file_name="Tool_Feedback.txt")
print(feedback_list)  # Should print the list of feedback records.
# fm.add_feedback("dft01", 4, "Not bad", "Game_Feedback.txt")
# fm.add_feedback("hi", 999, "Excel", "Game_Feedback.txt")
# feedback_list = fm.load_feedback(file_name="Game_Feedback.txt")
# print(feedback_list)

# # Check Subscription Manager Works.
subscriptions = sm.load_subscriptions()
print(sm.check_subscription("lbro", subscriptions))
# # Should return True or False based on the current date.
print(sm.get_rental_limit("Basic"))  # Should return 2.
print(sm.get_rental_limit("Premium"))  # Should return 7.

[{'ToolID': '0001', 'Rating': 4, 'Comments': 'amazing'}, {'ToolID': '0002', 'Rating': 2, 'Comments': 'too loud'}, {'ToolID': '0003', 'Rating': 5, 'Comments': 'Absolutely amazing!'}]
True
2
7


In [38]:
# HELPER: load grouped tools
def loadGroupedTools():
    rentals = loadRentalHistory()
    rentedIDs = { r['toolID'] for r in rentals if not r['returnDate'] or datetime.strptime(r['returnDate'], '%Y-%m-%d') > datetime.now() }
    catalog = {}
    with open("Tool_Info.txt", "r") as file:
        next(file)
        for line in file:
            line = line.strip()
            if line:
                id, name, size, pprice, pdate = line.split(',')
                key = (name, size)

                if key not in catalog:
                    catalog[key] = {
                        'name': name,
                        'size': size,
                        'purchasePrice': int(pprice),
                        'totalCopies': 0,
                        'allIDs': [],
                        'availableIDs': []
                    }

                catalog[key]['totalCopies'] += 1
                catalog[key]['allIDs'].append(id)
                if id not in rentedIDs:
                    catalog[key]['availableIDs'].append(id)

    return list(catalog.values())

In [39]:
# list all tools
def listTools(query=None, filter="all"):
    tools = loadGroupedTools()
    # if filter == "available":
    #     tools = [t for t in tools if isToolAvailable(t['id'])]
    if query:
        tools = [t for t in tools if query.lower() in t['name'].lower()]
    for tool in tools:
        print(f"• {tool['name']}; {len(tool['availableIDs'])}/{len(tool['allIDs'])} available")

In [40]:
# check if tool exists
def doesToolExist(toolID):
    tools = loadGroupedTools()
    for tool in tools:
        if toolID in tool['allIDs']:
            return True
    return False

In [41]:
# check if a customer has a subscription
def doesCustomerHaveSubscription(customerID):
    subscriptions = sm.load_subscriptions()
    return sm.check_subscription(customerID, subscriptions)

In [42]:
# HELPER: load rental history
def loadRentalHistory():
    rentals = []
    with open("Rental_History.txt", "r") as file:
        next(file)
        for line in file:
            line = line.strip()
            if line:
                toolID, customerID, rentalDate, returnDate = line.split(',')
                rentals.append({'toolID': toolID, 'customerID': customerID, 'rentalDate': rentalDate, 'returnDate': returnDate})
    return rentals

In [43]:
# check if a customer has reached their renting limit
def hasCustomerReachedRentingLimit(rentingCustomerID):
    count = 0
    rentals = loadRentalHistory()
    for rental in rentals:
        if rental['customerID'] == rentingCustomerID and (not rental['returnDate'] or datetime.strptime(rental['returnDate'], '%Y-%m-%d') > datetime.now()):
            count += 1
    subscriptions = sm.load_subscriptions()
    subscriptionPlan = subscriptions[rentingCustomerID]['SubscriptionType']
    return count >= sm.get_rental_limit(subscriptionPlan)

In [44]:
# check if tool is available
### Question: is the return date always logged when a tool is rented, or will this field remain empty until the tool is returned: same logic using in logToolReturn func
def isToolAvailable(toolID):
    if not doesToolExist(toolID):
        return False
    rentals = loadRentalHistory()
    for rental in rentals:
        if rental['toolID'] == toolID and (not rental['returnDate'] or datetime.strptime(rental['returnDate'], '%Y-%m-%d') > datetime.now()):
            return False
    return True

In [45]:
# log a tool rental
def logToolRental(toolID, customerID):
    flag = False
    with open("Rental_History.txt", "a") as file:
        file.write(f"{toolID},{customerID},{datetime.now().strftime('%Y-%m-%d')},\n")
        flag = True
    return flag

In [46]:
# log a tool returnal
def logToolReturnal(toolID):
    flag = False
    rentals = loadRentalHistory()
    active = list(filter(lambda rental: rental['toolID'] == toolID and (not rental['returnDate'] or datetime.strptime(rental['returnDate'], '%Y-%m-%d') > datetime.now()), rentals))

    if len(active) == 1:
        targetRental = active[0]

        with open("Rental_History.txt", "r+") as file:
            header = next(file)
            newLines = []

            for line in file:
                line = line.strip()
                if line:
                    rowToolID, rowCustomerID, rowRentalDate, rowReturnDate = line.split(',')
                    currentRow = {'toolID': rowToolID, 'customerID': rowCustomerID, 'rentalDate': rowRentalDate, 'returnDate': rowReturnDate}
                    if currentRow == targetRental:# and (not rowReturnDate or datetime.strptime(rowReturnDate, '%Y-%m-%d') > datetime.now()):
                        flag = True
                        currentRow['returnDate'] = datetime.now().strftime('%Y-%m-%d')
                    newLines.append(currentRow)

            file.seek(0)
            file.truncate()
            file.write(header)
            for row in newLines:
                file.write(f"{row['toolID']},{row['customerID']},{row['rentalDate']},{row['returnDate']}\n")
    return flag

In [47]:
# print(loadRentalHistory())
# print(loadTools())
# print(loadGroupedTools())
# logToolReturnal("3")

In [60]:
print("Welcome to the Tool Rental and DIY booking management system")
print("Please select from the following options")
opt = input("a) Check the availibility of tools\nr) Rent tools out to a customer\ne) Return a tool\nd) View a database of tools\ns) Search through all tools\n")

if opt == 'a':
    # check availbility of tools
    print("option a")

elif opt == 'r':
    # rent a tool out to a customer
    customerID = input("Which customer wishes to rent a tool? (Enter their ID)\n")

    # check if they have a valid subscription
    if doesCustomerHaveSubscription(customerID):
        print("Subscription confirmed")

        # check if they have reached their renting limit
        if not hasCustomerReachedRentingLimit(customerID):
            print("They have not reached their renting limit")
            toolID = input("Enter the tool ID they wish to rent out.")
            # will progress to UI with list of tools and select button so no tool id is needed

            # check if tool exists
            if doesToolExist(toolID):
                print("Tool exists")

                # check if tool is available
                if isToolAvailable(toolID):
                    print("Tool is available")

                    # log a tool rental
                    if logToolRental(toolID, customerID):
                        print("Tool rented")
                    else:
                        print("Error: Tool not rented")

                else:
                    print("Tool is not available")
            else:
                print("Tool does not exist")
        else:
            print("They have reached their renting limit")
    else:
        print(f"{customerID} does not have an onging subscription, so they will not be able to rent out any tools.")

elif opt == 'e':
    # return a tool
    print("Return a tool")
    toolID = input("Enter the tool ID they wish to return.")

    # log a tool returnal
    if logToolReturnal(toolID):
        print("Tool returned")

        # record tool feedback
        print("We wish to collect some feedback on the tool you just returned.")
        rating = int(input("Enter a star rating out of 5.\n"))
        comments = str(input("Enter some comments.\n"))
        fm.add_feedback(toolID, rating, comments, file_name="Tool_Feedback.txt")
        print("Thank you")

    else:
        print("Error: Tool not returned")

elif opt == 'd':
    # listing tools
    print("Here are all of the tools we have: ")
    listTools()

elif opt == 's':
    # searching tools
    query = input("Enter a search term.\n")
    print("Here are the tools that match your search critera: ")
    listTools(query=query)

else:
    print("Invalid option")

Welcome to the Tool Rental and DIY booking management system
Please select from the following options
a) Check the availibility of tools
r) Rent tools out to a customer
e) Return a tool
d) View a database of tools
s) Search through all tools
s
Enter a search term.
bu
Here are the tools that match your search critera: 
• Bulldozer; 0/1 available


In [49]:
import secrets
import string

def generate_id(length=4):
    alphabet = string.ascii_letters + string.digits
    return ''.join(secrets.choice(alphabet) for _ in range(length))

### TODO
# Cut lines to 80 chars max, via colab divider line